In [1]:
import sys
import os

sys.path.append(os.getcwd())
# append the root directory to the sys.path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

### Define Synthetic Dataset
(random - for now)

In [56]:
import numpy as np
import torch


# create synthetic dataset
class SynteticDataset:
    def __init__(self, num_samples: int= 1000, input_size: tuple= (64, 12), num_classes: int= 10):
        self.num_samples = num_samples
        self.input_size = input_size
        self.num_classes = num_classes
        self.data = self._generate_data()
    

    def _generate_data(self) -> tuple:
        X = torch.randint(0, 256, (self.num_samples, *self.input_size), dtype=torch.float32)
        y = torch.randint(0, self.num_classes, size=(self.num_samples,), dtype=torch.long)
        return X, y

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        X, y = self.data
        return X[idx], y[idx]

# define device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


### Initialize simple dataloaders

In [57]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split


NUM_SAMPLES = 100
INPUT_SIZE = (64, 64, 3) # e.g., mimicing 64x64 RGB images
NUM_CLASSES = 5


dataset = SynteticDataset(num_samples=NUM_SAMPLES, input_size=INPUT_SIZE, num_classes=NUM_CLASSES)
# split train test dataset
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle= True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle= False)

In [58]:
import matplotlib.pyplot as plt
img = train_dataset.dataset[0][0].numpy() / 255.0 # convert to ndarray and normalize
# plt.imshow(img)
# plt.show()

In [59]:
# Initialize Perceiver model
from src.models.perceiver import Perceiver
import torch.optim as optim
import torch.nn as nn


NUM_FREQ_BANDS = 6
DEPTH = 4
MAX_FREQ = 12
LATENT_HEADS = 2
NUM_LATENTS = 4
LATENT_DIM = 32
CROSS_HEADS = 4
INPUT_CHANNELS = 3

model = Perceiver(num_freq_bands= NUM_FREQ_BANDS,
                  depth= DEPTH,
                  max_freq= MAX_FREQ,
                  latent_heads= LATENT_HEADS,
                  cross_heads= CROSS_HEADS,
                  input_channels= INPUT_CHANNELS,
                  num_classes= NUM_CLASSES).to(device)

# Define optimizer and loss function


opti = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [60]:
def simple_train(train_loader, test_loader, device, model, criterion, optimizer, epochs= 5):
    logs = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    for epoch in range(epochs):
        model.train()
        temp_loss = 0.0
        temp_acc = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            temp_loss += loss.item() 
            temp_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)

        epoch_loss = temp_loss / len(train_loader)
        epoch_acc = temp_acc / len(train_loader)
        print(f'Epoch {epoch+1}/{epochs}, Training Loss: {epoch_loss:.4f}, Training Accuracy: {epoch_acc:.4f}')

        # Evaluate on test set
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        print(f'Epoch {epoch+1}/{epochs}, Test Accuracy: {accuracy:.2f}%')
        logs["train_loss"].append(epoch_loss)
        logs["train_acc"].append(epoch_acc)
        logs["test_acc"].append(accuracy)
    return logs
logs = simple_train(train_loader, test_loader, device, model = model, criterion = criterion, optimizer = opti, epochs= 5)


Epoch 1/5, Training Loss: 3.9206, Training Accuracy: 0.2375
Epoch 1/5, Test Accuracy: 0.00%
Epoch 2/5, Training Loss: 2.1771, Training Accuracy: 0.1750
Epoch 2/5, Test Accuracy: 0.40%
Epoch 3/5, Training Loss: 1.6758, Training Accuracy: 0.2125
Epoch 3/5, Test Accuracy: 0.00%
Epoch 4/5, Training Loss: 1.6647, Training Accuracy: 0.2250
Epoch 4/5, Test Accuracy: 0.25%
Epoch 5/5, Training Loss: 1.6765, Training Accuracy: 0.1750
Epoch 5/5, Test Accuracy: 0.00%
